# Chapter 2: LLM and Tool Use

## Introduction

You might be curious how AI agents are so powerful and can execute code—'do' everything for you.

Under the hood, an LLM is just a language model—it can only chat and cannot run anything. Think of it this way: you hired a personal tutor and provided the textbook, so the tutor understands what kind of algorithms you learned. These are all your tools. Given a question, your tutor tells you which 'tools' you should use and how to combine them, but your tutor will not actually solve it for you. You are the one who follows the tutor's suggestions and does everything.

So the LLM acts as the tutor: you provide the LLM with all the tools you have, and the LLM tells you which tools to use and how to use them (generates arguments). You then run the code. The agent framework coordinates all these steps—e.g., we can use Python exec to execute the code the LLM writes. By putting all these together, it appears as if the AI agent can not only think (LLM reasoning) but also take action (we use Python code to invoke the function the LLM chooses).

In this tutorial, we will deep dive into tool use using LangChain as an example.

## Table of Contents

[**Part 1: How LLMs Become Aware of Tools**](#part1)
- 1.1 Without Tool Information
- 1.2 Binding Tools to LLM

[**Part 2: How Functions Are Invoked**](#part2)
- 2.1 Manual Invocation (Plain Python)
- 2.2 Using LangChain's Tool.invoke()

[**Part 3: Agent Framework Coordinates Everything**](#part3)

[**Part 4: Under the Hood**](#part4)
- 4.1 Tool Binding Implementation
- 4.2 Tool Decorator Implementation



## <a id="part1">Part 1: How LLMs Become Aware of Tools </a>

As explained in the introduction, to use tools we first need to provide tool information to the LLM so it can decide what to do. In LangChain, we use `.bind_tools()` for this.

### 1.1 Without Tool Information

Let's first see what happens when the LLM doesn't know about our tools:

In [2]:
# we first define a model
import os
from langchain.tools import tool
from langchain.chat_models import init_chat_model

def plus_py(a: int, b: int) -> int:
    """Add two numbers together. input are two numbers, output is the sum"""
    return a + b

def multiply_py(a: int, b: int) -> int:
    """multiply two numbers together. input are two numbers, output is the product"""
    return a * b

# this one we will use to show in general how to invoke a function LLM decide to run
all_my_tools = {'plus':plus_py,
              "multiplication_two_numbers":multiply_py}

# Setup connection parameters
base_url = os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234/v1")
api_key = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,
    output_version= "responses/v1"
)

# This model currently have no tools
reply = model.invoke('hello AI, what is 45+69')
print('AI reply',reply.text)
print('This model is not aware of the tools we defined, print out the model reply you can observe this')
print('Any tool involved',reply.tool_calls)

AI reply 45 + 69 = **114**
This model is not aware of the tools we defined, print out the model reply you can observe this
Any tool involved []


<a id="part1-2"></a>

### 1.2 Binding Tools to LLM

Now we can bind the tool to let the model become aware of it.

#### Why is the above model not aware of the tools?

We can check in LM Studio—below is the request the model received. Notice there is no tool information:

```json
Received request: POST to /v1/responses with body  {
  "input": [
    {
      "content": "hello AI, what is 45+69",
      "role": "user"
    }
  ],
  "model": "openai/gpt-oss-20b",
  "stream": false
}
```

The model doesn't know you have a tool called `plus`, so it just works by itself and computes the answer.

#### Providing tool information to the model

In LangChain, you need to use the `@tool` decorator so it can generate tool information for the LLM to use. See the [Under the Hood](#part4) section for details.

In [41]:
@tool
def plus(a: int, b: int) -> int:
    """Add two numbers together. input are two numbers, output is the sum"""
    return a + b

@tool("multiplication_two_numbers") # this example shows set a name
def multiply(a: int, b: int) -> int:
    """multiply two numbers together. input are two numbers, output is the product"""
    return a * b

#these are tools modified by langchain @tool
all_my_langchain_tools = {'plus':plus,
              "multiplication_two_numbers":multiply}

# we bind the tools to the model, this will give use a new model
model_with_tools = model.bind_tools(tools=[plus])
reply = model_with_tools.invoke('hello AI, what is 45+69')

print('AI reply',reply.text)

print('Any tool involved',reply.tool_calls)

AI reply 
Any tool involved [{'name': 'plus', 'args': {'a': 45, 'b': 69}, 'id': 'call_3149652188900153', 'type': 'tool_call'}]


#### What the LLM receives with tool binding

If you examine your LM Studio console, you'll find that the request now includes function information:

```json
Received request: POST to /v1/responses with body  {
  "input": [
    {
      "content": "hello AI, what is 45+69",
      "role": "user"
    }
  ],
  "model": "openai/gpt-oss-20b",
  "stream": false,
  "tools": [
    {
      "type": "function",
      "name": "plus",
      "description": "Add two numbers together. input are two numbers, output is the sum",
      "parameters": {
        "properties": {
          "a": {
            "type": "integer"
          },
          "b": {
            "type": "integer"
          }
        },
        "required": [
          "a",
          "b"
        ],
        "type": "object"
      }
    }
  ]
}
```

Notice the `tools` array that describes the function schema!


## <a id="part2">Part 2: How Functions Are Invoked</a>



So now we know that if you provide tool information to the LLM, it can select which tools to use. The next question is: **how is the function actually invoked?**

Let's explore two different approaches:



### <a id="part2-1">2.1 Manual Invocation (Plain Python)</a>

We can parse the tool call information and invoke the function as a regular Python function:

In [42]:
# we can first parse out the function name
func_name = reply.tool_calls[0]['name']
arg = reply.tool_calls[0]['args']

print(f'func_name: {func_name}, arg: {arg}')

# now we get real function
func = all_my_tools[func_name]
# we just run it as any python function
print(func(**arg))

func_name: plus, arg: {'a': 45, 'b': 69}
114


<a id="part2-2"></a>

### 2.2 Using LangChain's Tool.invoke()

You can also use LangChain's tool wrapper. The `@tool` decorator modifies the function, and you need to use `.invoke()`:



In [43]:
# Get the function from our langchain tool dictionary
func = all_my_langchain_tools[func_name]

result = func.invoke(arg)
print(f'Result: {result}')

Result: 114




## <a id="part3">Part 3: Agent Framework Coordinates Everything</a>

The agent framework makes it easy for you to develop AI systems by coordinating all the steps automatically. We can redo the above using LangChain's agent framework.

In the example below, using `create_agent` from LangChain, after the user asks a question, these events happen sequentially:

1. **LLM decides** whether and how to call a tool
2. **Tool is automatically invoked** by the framework
3. **Tool results are fed back to the LLM**, and the LLM decides how to answer

This is the key difference: you don't manually parse tool calls or invoke functions—the agent does it all for you!

In [44]:
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def plus(a: int, b: int) -> int:
    """Add two numbers together. input are two numbers, output is the sum"""
    return a + b

@tool("multiplication_two_numbers")  # sets the tool name explicitly
def multiply(a: int, b: int) -> int:
    """multiply two numbers together. input are two numbers, output is the product"""
    return a * b

agent = create_agent(
    model=model,
    tools=[plus, multiply],
    system_prompt="You are a helpful assistant. Use tools for math when helpful.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "hello AI, what is 45+69"}]}
)

# create_agent returns a state dict; the final assistant message is usually last
for idx, msg in enumerate(result['messages']):
    print('='*20,'message No.',idx,'='*20)
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f'  - AI decided to use: {msg.tool_calls}')
    else:
        print(f'  - {msg.type}: {msg.text}')
print('='*20,'*end of message')

==================== message No. 0 ====================
  - human: hello AI, what is 45+69
==================== message No. 1 ====================
  - AI decided to use: [{'name': 'plus', 'args': {'a': 45, 'b': 69}, 'id': 'call_3149652188900154', 'type': 'tool_call'}]
==================== message No. 2 ====================
  - tool: 114
==================== message No. 3 ====================
  - ai: The sum of 45 and 69 is **114**.
==================== *end of message


### What Else Does the Agent Framework Handle?

Agent frameworks also handle other important things such as:
- **Error handling**: Retry logic, fallback strategies
- **Persistence**: Saving conversation state
- ..........
..........

LangChain's agent is built on top of LangGraph. We will deep dive into these topics after we finish covering the key concepts in LangGraph.

## <a id="part4">Part 4: Under the Hood</a>

It is worth looking at how LangChain binds tools to LLMs and how the `@tool` decorator works internally.

<a id="part4-1"></a>

### 4.1 Tool Binding Implementation

In `langchain_core/runnables/base.py`, you can find:

```python
class RunnableBinding(RunnableBindingBase[Input, Output]):
    # --- other code ---
    
    @override
    def bind(self, **kwargs: Any) -> Runnable[Input, Output]:
        """Bind additional kwargs to a Runnable, returning a new Runnable.

        Args:
            **kwargs: The kwargs to bind to the Runnable.

        Returns:
            A new Runnable with the same type and config as the original,
            but with the additional kwargs bound.
        """
        return self.__class__(
            bound=self.bound,
            config=self.config,
            config_factories=self.config_factories,
            kwargs={**self.kwargs, **kwargs},
            custom_input_type=self.custom_input_type,
            custom_output_type=self.custom_output_type,
        )
```

**Key Design Choice**: This approach allows you to fix some kwargs at call time. It's more flexible than having a class method like `from_tool()` because this approach allows **multiple binding events to be chained together**—each time you get the same runnable type, you can continue binding additional kwargs.

<a id="part4-2"></a>

### 4.2 Tool Decorator Implementation

The `@tool` decorator is defined in `langchain_core/tools/convert.py`. When you use `@tool`, it wraps your function in a `StructuredTool` class:

```python
def _create_tool_factory(tool_name: str) -> Callable[[Union[Callable, Runnable]], BaseTool]:
    def _tool_factory(dec_func: Union[Callable, Runnable]) -> BaseTool:
        return StructuredTool.from_function(...)
    
    return _tool_factory
```

This is why `all_my_langchain_tools['plus']` is not just a plain function, but a `StructuredTool` object with additional methods like `.invoke()`.

#### Schema Generation

Another important aspect is how the tool schema (the tool information we feed into the LLM) is generated. The schema requires either a docstring, description, or explicit schema. The schema is generated by calling class methods like `StructuredTool.from_function()` (from `structured.py`):

```python
if infer_schema or args_schema is not None:
    return StructuredTool.from_function(
        func,
        coroutine,
        name=tool_name,
        description=tool_description,
        return_direct=return_direct,
        args_schema=schema,
        infer_schema=infer_schema,
        response_format=response_format,
        parse_docstring=parse_docstring,
        error_on_invalid_docstring=error_on_invalid_docstring,
    )
# If someone doesn't want a schema applied, we must treat it as
# a simple string->string function
if dec_func.__doc__ is None:
    msg = (
        "Function must have a docstring if "
        "description not provided and infer_schema is False."
    )
    raise ValueError(msg)
```

The schema itself is generated using Pydantic's validation (from `langchain_core/tools/base.py`):

```python
def create_schema_from_function(
    model_name: str,
    func: Callable,
    *,
    parse_docstring: bool = False,
    error_on_invalid_docstring: bool = False,
) -> Type[BaseModel]:
    from pydantic import validate_arguments

    validated = validate_arguments(func)
    inferred_model = validated.model

    # Remove self/cls and injected args
    # ...
    return inferred_model
```

This is how LangChain automatically infers the tool schema from your function's type hints!